# 03 — Feature Engineering
**Week 2, Steps 8–10**

Builds the Analytical Base Table (one row per order), engineers 15+ features,
and builds customer-level features as-of a date.

> Run **Restart & Run All** to reproduce from scratch.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 4)

from src.ingest import load_all
from src.clean import CleaningLog, DataCleaner, CITY_ALIASES
from src.features import build_analytical_base_table, add_features, build_customer_features

rng = np.random.default_rng(42)  # seeded RNG for reproducibility (PRD requirement)
pd.set_option('display.max_columns', 20)

print("Libraries loaded ✅")


## 1. Build cleaned tables

In [ ]:
raw         = load_all()
cities_list = raw['cities']['city'].dropna().str.strip().tolist()
log         = CleaningLog()
cleaner     = DataCleaner(cities_list, CITY_ALIASES, log)
cleaned     = cleaner.clean_all(raw)
print(f"Cleaned: {len(cleaned)} tables  |  orders: {len(cleaned['orders']):,}")


## 2. Build Analytical Base Table

In [ ]:
n_orders = len(cleaned['orders'])
abt_raw = build_analytical_base_table(cleaned)
assert len(abt_raw) == n_orders, f"Row count changed! {n_orders} → {len(abt_raw)}"
print(f"ABT: {len(abt_raw):,} rows × {len(abt_raw.columns)} cols  (one row per order ✅)")


## 3. Engineer features

In [ ]:
abt = add_features(abt_raw)
assert len(abt) == n_orders
print(f"ABT with features: {len(abt):,} rows × {len(abt.columns)} cols")


## 4. Feature catalogue

In [ ]:
feature_catalogue = [
    ("peak_hour",                  "Binary 0/1 — order placed in lunch (12-14) or dinner (19-22) peak"),
    ("weekend_order",              "Binary 0/1 — order placed on Saturday or Sunday"),
    ("rain_impact",                "Ordinal: None/Light/Moderate/Heavy — from rainfall mm"),
    ("traffic_score",              "Numeric 1-4 (Low/Moderate/High/Severe) — from traffic join on city+date+hour"),
    ("customer_tenure_days",       "Days between customer registration and this order"),
    ("restaurant_popularity_rank", "Dense rank of restaurant by order count within its city"),
    ("delivery_efficiency",        "Partner avg_delivery_time ÷ city average — below 1 = faster than city"),
    ("average_rating",             "Mean of 3 feedback ratings — EDA/BI only, NOT a delivery-time model input"),
    ("n_items",                    "Total items in this order (sum of quantity from order_items)"),
    ("discount_pct",               "discount ÷ food_cost — fraction of order discounted"),
    ("partner_tenure_days",        "Days from partner joining_date to order_date"),
    ("is_peak_and_rain",           "Interaction: peak_hour=1 AND rain_impact in Moderate/Heavy"),
    ("is_weekend_and_rain",        "Interaction: weekend_order=1 AND rain_impact in Moderate/Heavy"),
    ("coupon_used",                "Binary 0/1 — whether a coupon was applied"),
    ("distance_to_city_centre_km", "Haversine km from restaurant to mean lat/lon of city's restaurants (proxy — no customer coords)"),
    ("is_cross_city",              "Binary 0/1 — customer city ≠ restaurant city"),
    ("avg_prep_time",              "Mean preparation time (mins) across items in this order"),
    ("max_prep_time",              "Max preparation time (mins) across items in this order"),
    ("dt_outlier_flag",            "IQR k=3.0 outlier flag on delivery_time_minutes — flagged, not capped"),
]

pd.DataFrame(feature_catalogue, columns=['feature','description'])


## 5. Feature distributions — key plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 5a: Delivery time by PeakHour
peak_data = [
    abt.loc[abt['peak_hour']==0,'delivery_time_minutes'].dropna(),
    abt.loc[abt['peak_hour']==1,'delivery_time_minutes'].dropna()
]
axes[0,0].boxplot(peak_data, labels=['Off-peak','Peak'])
axes[0,0].set_title('Delivery Time by PeakHour')
axes[0,0].set_ylabel('minutes')

# 5b: Delivery time by WeekendOrder
wk_data = [
    abt.loc[abt['weekend_order']==0,'delivery_time_minutes'].dropna(),
    abt.loc[abt['weekend_order']==1,'delivery_time_minutes'].dropna()
]
axes[0,1].boxplot(wk_data, labels=['Weekday','Weekend'])
axes[0,1].set_title('Delivery Time by WeekendOrder')
axes[0,1].set_ylabel('minutes')

# 5c: Delivery time by RainImpact
rain_order = ['None','Light','Moderate','Heavy']
rain_data = [abt.loc[abt['rain_impact']==r,'delivery_time_minutes'].dropna()
             for r in rain_order]
axes[0,2].boxplot([d for d in rain_data if len(d)>0],
                  labels=[r for r, d in zip(rain_order, rain_data) if len(d)>0])
axes[0,2].set_title('Delivery Time by RainImpact')
axes[0,2].set_ylabel('minutes')

# 5d: Delivery efficiency histogram
axes[1,0].hist(abt['delivery_efficiency'].dropna(), bins=40, color='purple', edgecolor='white')
axes[1,0].axvline(1.0, color='red', linestyle='--', label='City average')
axes[1,0].set_title('Delivery Efficiency (partner vs city avg)')
axes[1,0].legend()

# 5e: Delivery time by order_hour (hourly median)
hourly = abt.groupby('order_hour')['delivery_time_minutes'].median()
axes[1,1].plot(hourly.index, hourly.values, marker='o', color='darkorange')
axes[1,1].set_title('Median Delivery Time by Hour')
axes[1,1].set_xlabel('Hour of day')
axes[1,1].set_ylabel('minutes')

# 5f: Peak hour share
peak_counts = abt['peak_hour'].value_counts().sort_index()
axes[1,2].bar(['Off-peak','Peak'], peak_counts.values, color=['steelblue','coral'])
axes[1,2].set_title(f'Peak Hour Orders ({abt["peak_hour"].mean()*100:.1f}% peak)')

plt.tight_layout()
plt.savefig('reports/feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved → reports/feature_distributions.png")


## 6. GroupBy, pivot, and cross-tab summaries (PRD NumPy/Pandas requirements)

In [ ]:
# PRD requirement: cross-tabulation
print("=== Cross-tab: membership × order_status ===")
ct = pd.crosstab(abt['membership'], abt['order_status'])
print(ct)


In [ ]:
# PRD requirement: pivot table
print("\n=== Pivot: restaurant_city × order_status (mean delivery time) ===")
pv = abt.pivot_table(
    values='delivery_time_minutes',
    index='restaurant_city',
    columns='order_status',
    aggfunc='mean'
).round(1)
pv.head(5)


In [ ]:
# PRD requirement: GroupBy summary
print("=== GroupBy: avg delivery time by vehicle_type ===")
abt.groupby('vehicle_type')['delivery_time_minutes'].agg(['mean','median','count']).round(2)


## 7. Categorical encoding demo (ordinal + get_dummies)

In [ ]:
# Ordinal encoding for traffic_level (already done as traffic_score)
# Demo pd.get_dummies for nominal columns — real encoding goes in Week 3 pipeline
sample = abt[['membership','order_status','restaurant_cuisine']].head(5)
print("One-hot encoding demo (real encoding in Week 3 sklearn pipeline):")
pd.get_dummies(sample, columns=['membership','order_status'], drop_first=True)


## 8. CustomerTenure negative check

In [ ]:
neg_tenure = abt['customer_tenure_days'] < 0
print(f"Orders with negative customer tenure (order before registration): {neg_tenure.sum():,}")
print("These rows are flagged via 'tenure_is_negative' column and will be")
print("excluded from model features in Week 3 (not from the dataset entirely).")
abt[neg_tenure][['order_id','customer_id','order_date','registration_date','customer_tenure_days']].head()


## 9. Percentile stats (PRD NumPy requirement)

In [ ]:
valid_dt = abt['delivery_time_minutes'].dropna()
pcts = np.percentile(valid_dt, [50, 75, 90, 95, 99, 99.9])
print("Delivery time percentiles (valid orders):")
for p, v in zip([50,75,90,95,99,99.9], pcts):
    print(f"  p{p:<5}  {v:.1f} min")
print(f"\nLate delivery threshold (p75): {pcts[1]:.0f} min")
print("→ This matches kpi_definitions.md: 'late' = delivery_time_minutes > 45")


## 10. Customer features as-of reference date (Step 10)

In [ ]:
orders_clean = cleaned['orders'].copy()
orders_clean['order_date']   = pd.to_datetime(orders_clean['order_date'], errors='coerce')
orders_clean['final_amount'] = pd.to_numeric(orders_clean['final_amount'], errors='coerce')

ref_date = orders_clean['order_date'].max()
cust_features = build_customer_features(orders_clean, as_of=ref_date)

print(f"Reference date: {ref_date.date()} (never CURRENT_DATE — the data is a fixed snapshot)")
print(f"Customers with at least 1 delivered order: {len(cust_features):,}")
print(f"Active (≤60 days since last order): {(cust_features['days_since_last_order']<=60).sum():,}")
print(f"Churned (>60 days): {(cust_features['days_since_last_order']>60).sum():,}\n")
print("Why the cut-off date matters:")
print("  Week 3 will call build_customer_features(orders, as_of=cutoff) with an")
print("  earlier date, then label customers as churned if they didn't order in")
print("  the following 60 days. If we used all data, the model would see the")
print("  future orders it is supposed to predict.")
cust_features.describe().round(2)


## 11. Distance limitation note

In [ ]:
print("=== Distance feature: honest proxy ===")
print()
print("customers.csv has no latitude/longitude columns (verified in Step 9).")
print("The PRD's 'restaurant-to-customer distance' was replaced with:")
print()
print("  distance_to_city_centre_km = haversine_km(")
print("      restaurant.lat, restaurant.lon,")
print("      mean(lat for all restaurants in same city),")
print("      mean(lon for all restaurants in same city)")
print("  )")
print()
print("This is a proxy, not the actual delivery distance.")
print("Documented in documentation/kpi_definitions.md and validation_report.md.")
print()
dist = abt['distance_to_city_centre_km'].dropna()
print(f"  min={dist.min():.1f} km  median={dist.median():.1f} km  max={dist.max():.1f} km")


## 12. Final ABT shape

In [ ]:
print(f"=== Analytical Base Table ===")
print(f"  Rows:    {len(abt):,}  (one per cleaned order ✅)")
print(f"  Columns: {len(abt.columns)}")
print()
print("Column groups:")
for label, cols in [
    ("Order",        [c for c in abt.columns if c.startswith("order_") or c in ["order_id","final_amount","food_cost","delivery_fee","gst","coupon_code","discount","payment_method","order_status","dt_outlier_flag"]]),
    ("Customer",     [c for c in abt.columns if c.startswith("customer_") or c in ["membership","registration_date","state","preferred_cuisine","total_orders","tenure_is_negative","customer_tenure_days"]]),
    ("Restaurant",   [c for c in abt.columns if c.startswith("restaurant_") or c in ["area","average_cost","latitude","longitude","city_lat","city_lon","distance_to_city_centre_km","is_cross_city","restaurant_order_count","restaurant_popularity_rank"]]),
    ("Partner",      [c for c in abt.columns if c.startswith("partner_") or c in ["vehicle_type","completed_deliveries","delivery_efficiency"]]),
    ("Features",     ["peak_hour","weekend_order","rain_impact","traffic_score","is_peak_and_rain","is_weekend_and_rain","coupon_used","order_hour","order_minute","order_dayofweek","order_month"]),
    ("Basket",       ["n_items","n_distinct_items","basket_value","avg_prep_time","max_prep_time","discount_pct"]),
    ("Weather",      ["temperature","rainfall","humidity","weather_condition"]),
    ("Traffic",      ["traffic_level","traffic_score","average_speed"]),
    ("Feedback",     [c for c in abt.columns if c.startswith("feedback_") or c in ["review","sentiment","has_review"]]),
]:
    found = [c for c in cols if c in abt.columns]
    if found:
        print(f"  {label:<12} ({len(found):>2}): {', '.join(found[:6])}{'...' if len(found)>6 else ''}")
